
# Identification of Amyloid-β Oligomers from MD Trajectories

This notebook identifies amyloid-β (Aβ) oligomer configurations from all-atom molecular dynamics (MD) trajectories using peptide–peptide contact networks.

## Methodology

Two peptides are considered connected if at least one pair of heavy atoms lies within a specified distance cutoff (default: 0.35 nm). Connected peptide clusters are identified using a graph-based connectivity algorithm.

An oligomer of size **N** is identified when exactly N peptides belong to the same connected cluster. Direct contact between every peptide pair is not required; indirect connections are sufficient.

The analysis includes:

- Identification of peptide heavy atoms
- Construction of peptide–peptide contact matrices
- Identification of connected oligomer clusters
- Detection of frames containing a specified oligomer size
- Calculation of oligomer fractions across trajectory frames
- Generation of PDB file lists for identified configurations

## Analysis Parameters

The user can specify:

- **`n_peptides`**: Total number of peptides in the simulation
- **`oligomer_size`**: Target oligomer size (monomer, dimer, trimer, etc.)
- **`cutoff_nm`**: Heavy-atom contact distance cutoff
- **`start_ns`, `end_ns`**: Analysis time interval
- **`pdb_dt_ps`**: Sampling interval for trajectory frames

## Output

For each independent trajectory, the notebook reports:

- Total number of frames analyzed
- Number of frames containing the target oligomer
- Fraction of frames containing the target oligomer

The fraction is calculated as:

$$
f_N = \frac{\text{Frames containing an oligomer of size } N}{\text{Total frames analyzed}}
$$

The notebook also generates a text file listing the corresponding PDB paths. PDB files must be generated separately using the same trajectory sampling and frame-numbering convention.

### Important Distinction: Oligomer Counts vs. Frame Counts

This analysis counts **frames containing at least one connected cluster of exactly the specified oligomer size**, rather than the total number of individual oligomers.

For example, in a system containing six peptides:

- A frame containing two separate trimers counts as **one trimer-containing frame**, not two oligomers.
- A frame containing a tetramer and a dimer counts as **one tetramer-containing frame and one dimer-containing frame** when those sizes are analyzed separately.
- A frame containing one fully connected hexamer counts as a **hexamer-containing frame**, but not as a dimer-, trimer-, tetramer-, or pentamer-containing frame.

Thus, the reported fraction represents the frequency of frames containing the specified oligomer size, not the number or concentration of oligomeric species.

The generated PDB paths correspond to complete simulation frames, not isolated oligomer structures.

## Example System

The example analyzes five independent trajectories containing six Aβ40 peptides, 21 DMPC molecules, and 9 cholesterol molecules.

Frames are sampled from 600–1000 ns at 1 ns intervals using a heavy-atom contact cutoff of 0.35 nm.

The target oligomer size can be changed without modifying the underlying contact calculation.


In [1]:

import mdtraj as md
import numpy as np
import os
from itertools import combinations


def connected_cluster_sizes(contact_matrix):
    """
    Identify the sizes of all connected peptide clusters
    based on direct or indirect contacts.
    """
    visited = set()
    cluster_sizes = []

    for start_node in range(contact_matrix.shape[0]):

        if start_node in visited:
            continue

        stack = [start_node]
        cluster_size = 0

        while stack:
            node = stack.pop()

            if node not in visited:
                visited.add(node)
                cluster_size += 1

                for neighbor in range(contact_matrix.shape[0]):
                    if contact_matrix[node, neighbor] == 1 and neighbor not in visited:
                        stack.append(neighbor)

        cluster_sizes.append(cluster_size)

    return cluster_sizes


def get_peptide_heavy_atoms(topology, residues_per_peptide, n_peptides=6):
    """
    Identify heavy-atom indices for each peptide, assuming
    peptides are ordered consecutively in the topology.
    """
    protein_residues = [res for res in topology.residues if res.is_protein]

    expected = residues_per_peptide * n_peptides
    if len(protein_residues) < expected:
        raise ValueError(
            f"Expected at least {expected} protein residues, "
            f"but found {len(protein_residues)}"
        )

    peptide_atoms = []

    for peptide_id in range(n_peptides):
        start = peptide_id * residues_per_peptide
        end = (peptide_id + 1) * residues_per_peptide

        atoms = []
        for residue in protein_residues[start:end]:
            for atom in residue.atoms:
                if atom.element is None or atom.element.symbol != "H":
                    atoms.append(atom.index)

        peptide_atoms.append(np.array(atoms, dtype=int))

    return peptide_atoms


In [2]:

def make_oligomer_pdb_list(
    trajectory_files, topology_files, replica_ids, pdb_directory,
    residues_per_peptide, output_list_file, start_ns=600, end_ns=1000,
    pdb_dt_ps=1000, cutoff_nm=0.35, n_peptides=6,
    oligomer_size=6, pdb_prefix="protein"
):
    """
    Identify oligomer configurations of a specified size from MD trajectories.

    Two peptides are connected when any pair of heavy atoms is within
    the specified cutoff. An oligomer is identified when a connected
    cluster contains exactly oligomer_size peptides.

    Parameters
    ----------
    n_peptides : int
        Total number of peptides in the simulation.
    oligomer_size : int
        Size of the oligomer to identify (e.g., 2, 3, 4, 5, or 6).

    Returns
    -------
    all_pdb_paths : list of str
        PDB paths corresponding to frames containing the target oligomer.
    summary : list of dict
        Frame counts and oligomer fractions for each trajectory.
    """

    if not 1 <= oligomer_size <= n_peptides:
        raise ValueError("oligomer_size must be between 1 and n_peptides.")

    start_ps = start_ns * 1000
    end_ps = end_ns * 1000
    tolerance = 1e-3

    all_pdb_paths = []
    summary = []

    for traj_file, top_file, replica_id in zip(trajectory_files, topology_files, replica_ids):

        topology = md.load_topology(top_file)
        peptide_atoms = get_peptide_heavy_atoms(
            topology, residues_per_peptide=residues_per_peptide, n_peptides=n_peptides
        )

        pdb_frame_index = 0
        frames_checked = 0
        oligomer_count = 0

        print(f"\nProcessing replica {replica_id}")

        for chunk in md.iterload(traj_file, top=top_file):

            times_ps = chunk.time
            k = np.rint((times_ps - start_ps) / pdb_dt_ps)
            expected_times = start_ps + k * pdb_dt_ps

            time_mask = (
                (times_ps >= start_ps - tolerance) &
                (times_ps <= end_ps + tolerance) &
                (np.abs(times_ps - expected_times) < tolerance)
            )

            if not np.any(time_mask):
                continue

            selected_chunk = chunk[time_mask]

            for local_frame in range(selected_chunk.n_frames):

                frame = selected_chunk[local_frame]
                contact_matrix = np.zeros((n_peptides, n_peptides), dtype=int)

                for i, j in combinations(range(n_peptides), 2):

                    neighbors = md.compute_neighbors(
                        frame, cutoff_nm,
                        query_indices=peptide_atoms[i],
                        haystack_indices=peptide_atoms[j],
                        periodic=True
                    )

                    if len(neighbors[0]) > 0:
                        contact_matrix[i, j] = 1
                        contact_matrix[j, i] = 1

                frames_checked += 1

                # Identify frames containing the specified oligomer size.
                cluster_sizes = connected_cluster_sizes(contact_matrix)

                if oligomer_size in cluster_sizes:
                    pdb_name = f"{pdb_prefix}_{replica_id}_{pdb_frame_index}.pdb"
                    pdb_path = os.path.join(pdb_directory, pdb_name)

                    all_pdb_paths.append(pdb_path)
                    oligomer_count += 1

                pdb_frame_index += 1

        fraction = oligomer_count / frames_checked if frames_checked > 0 else 0

        summary.append({
            "replica": replica_id,
            "frames_checked": frames_checked,
            "oligomer_frames": oligomer_count,
            "fraction_oligomer": fraction
        })

        print(f"Frames checked: {frames_checked}")
        print(f"Oligomer frames (size {oligomer_size}): {oligomer_count}")
        print(f"Fraction oligomer: {fraction:.3f}")

    with open(output_list_file, "w") as f:
        for pdb_path in all_pdb_paths:
            f.write(pdb_path + "\n")

    print("\nFinished.")
    print(f"Total frames containing oligomers of size {oligomer_size}: {len(all_pdb_paths)}")

    return all_pdb_paths, summary


In [3]:

from pathlib import Path

# Base directory and system
BASE_DIR = Path("/path/to/AB_lipids_1_5_ratio")
SYSTEM = "AB40_six_monomers_21DMPC_9CHL1"

# Oligomer size to identify (1–6)
OLIGOMER_SIZE = 6

# Input trajectories and topologies (5 independent replicas)
traj_dir = BASE_DIR / "xtc_files_for_movies" / SYSTEM

trajectory_files = [str(traj_dir / f"movie_smooth_{i:02d}.xtc") for i in range(1, 6)]
topology_files = [str(traj_dir / f"movie_smooth_{i:02d}.gro") for i in range(1, 6)]
replica_ids = [f"{i:02d}" for i in range(1, 6)]

# PDB directory and output file
pdb_directory = BASE_DIR / "600-1000_per_frame_pdb_files" / SYSTEM
output_dir = BASE_DIR / "contact_maps_600-1000" / SYSTEM
output_list_file = output_dir / f"input_pdbs_size_{OLIGOMER_SIZE}.txt"

# Identify oligomer configurations
pdb_paths, summary = make_oligomer_pdb_list(
    trajectory_files=trajectory_files,
    topology_files=topology_files,
    replica_ids=replica_ids,
    pdb_directory=str(pdb_directory),
    residues_per_peptide=40,
    output_list_file=str(output_list_file),
    start_ns=600,
    end_ns=1000,
    pdb_dt_ps=1000,
    cutoff_nm=0.35,
    n_peptides=6,
    oligomer_size=6,
    pdb_prefix="protein"
)



Processing replica 01
Frames checked: 401
Oligomer frames (size 6): 246
Fraction oligomer: 0.613

Processing replica 02
Frames checked: 401
Oligomer frames (size 6): 193
Fraction oligomer: 0.481

Processing replica 03
Frames checked: 401
Oligomer frames (size 6): 398
Fraction oligomer: 0.993

Processing replica 04
Frames checked: 401
Oligomer frames (size 6): 401
Fraction oligomer: 1.000

Processing replica 05
Frames checked: 401
Oligomer frames (size 6): 298
Fraction oligomer: 0.743

Finished.
Total frames containing oligomers of size 6: 1536
